# Seattle Bridge Closure Probability Analysis
This notebook analyzes the historical bridge closure data scraped from `@SDOTbridges` to determine the probability of a bridge being closed during any given 5-minute increment of the day.

We are specifically focusing on:
- **Weekdays only** (Monday - Friday)
- **Fremont Bridge** vs **Ballard Bridge**


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Handle different working directories by starting with the absolute path
file_path = '/Users/charliethompson/Documents/mosp/posts/bridgelocks/data/processed_bridge_openings.csv'

if not os.path.exists(file_path):
    file_path = '../data/processed_bridge_openings.csv'
if not os.path.exists(file_path):
    file_path = 'data/processed_bridge_openings.csv'
if not os.path.exists(file_path):
    file_path = 'processed_bridge_openings.csv' # Colab fallback

print(f"Loading data from: {file_path}")
df = pd.read_csv(file_path)

# Convert timestamps to datetime objects
df['start_time'] = pd.to_datetime(df['start_time'])
df['end_time'] = pd.to_datetime(df['end_time'])

# Filter for Fremont and Ballard bridges only
df = df[df['bridge'].isin(['Fremont', 'Ballard'])]

# Filter for Weekdays (Monday=0, Sunday=6)
# Keep only days 0 to 4
df = df[df['start_time'].dt.dayofweek < 5]

print(f"Total weekday bridge closures analyzed:\n{df['bridge'].value_counts().to_string()}")

# Calculate the total number of weekdays in our dataset's date range
min_date = df['start_time'].dt.date.min()
max_date = df['start_time'].dt.date.max()

total_weekdays = np.busday_count(min_date, max_date + timedelta(days=1))
print(f"\nTotal number of weekdays in the dataset (from {min_date} to {max_date}): {total_weekdays}")

Loading data from: processed_bridge_openings.csv


FileNotFoundError: [Errno 2] No such file or directory: 'processed_bridge_openings.csv'

In [ ]:
# Create 5-minute bins for a 24-hour day (288 bins total)
times = pd.date_range("00:00", "23:55", freq="5min").time

# Initialize counts
counts = {
    'Fremont': {t: 0 for t in times},
    'Ballard': {t: 0 for t in times}
}

# Iterate over all closure events
for _, row in df.iterrows():
    b = row['bridge']
    
    event_start_min = row['start_time'].hour * 60 + row['start_time'].minute
    event_end_min = row['end_time'].hour * 60 + row['end_time'].minute
    
    if event_end_min < event_start_min:
        segments = [(event_start_min, 1440), (0, event_end_min)]
    else:
        segments = [(event_start_min, event_end_min)]
        
    for t in times:
        bin_start_min = t.hour * 60 + t.minute
        bin_end_min = bin_start_min + 5
        
        overlap = False
        for seg_start, seg_end in segments:
            if seg_start < bin_end_min and seg_end > bin_start_min:
                overlap = True
                break
                
        if overlap:
            counts[b][t] += 1

probs_fremont = [counts['Fremont'][t] / total_weekdays for t in times]
probs_ballard = [counts['Ballard'][t] / total_weekdays for t in times]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Helper to split data and plot clocks
def plot_clock_face(ax, probs, title, color, max_prob=None):
    num_bins = len(probs)
    angles = np.linspace(0, 2 * np.pi, num_bins, endpoint=False)
    # Start at top (pi/2) and go clockwise
    width = (2 * np.pi) / num_bins

    # Plot the probability bars
    ax.bar(angles, probs, width=width, color=color, alpha=0.8, edgecolor='none', zorder=2)

    # Thematic clock settings
    ax.set_theta_offset(np.pi/2)
    ax.set_theta_direction(-1)

    # Set full 12-hour clock labels
    tick_labels = ['12', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11']
    tick_angles = np.linspace(0, 2 * np.pi, 12, endpoint=False)

    ax.set_xticks(tick_angles)
    ax.set_xticklabels(tick_labels, fontsize=12, fontweight='bold')

    # Draw to enable label positioning for rotation
    plt.draw()

    for label, angle in zip(ax.get_xticklabels(), tick_angles):
        rotation = np.degrees(-angle)
        label.set_rotation(rotation)
        label.set_va('center')
        label.set_ha('center')

    # Styling
    ax.set_title(title, fontsize=14, pad=35, fontweight='bold')
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.set_rlabel_position(0)

    if max_prob:
        ax.set_ylim(0, max_prob)

    # Visual elements: pin and thick border
    ax.plot(0, 0, marker='o', markersize=10, color='black', zorder=5)
    ax.spines['polar'].set_linewidth(3)
    ax.spines['polar'].set_color('#333333')
    ax.set_facecolor('#f9f9f9')

# Split 288 bins into 144 AM and 144 PM
am_fremont = probs_fremont[:144]
pm_fremont = probs_fremont[144:]
am_ballard = probs_ballard[:144]
pm_ballard = probs_ballard[144:]

global_max = max(max(probs_fremont), max(probs_ballard))

# Layout adjustments
fig, axes = plt.subplots(2, 2, figsize=(16, 16), subplot_kw={'projection': 'polar'})

plot_clock_face(axes[0,0], am_fremont, 'Fremont Bridge (AM)', '#1f77b4', max_prob=global_max)
plot_clock_face(axes[0,1], pm_fremont, 'Fremont Bridge (PM)', '#1f77b4', max_prob=global_max)
plot_clock_face(axes[1,0], am_ballard, 'Ballard Bridge (AM)', '#ff7f0e', max_prob=global_max)
plot_clock_face(axes[1,1], pm_ballard, 'Ballard Bridge (PM)', '#ff7f0e', max_prob=global_max)

plt.suptitle('Seattle Bridge Closure Probabilities\n(Linked Scale Analog Clocks)', fontsize=22, y=0.98, fontweight='bold')
plt.subplots_adjust(hspace=0.4, wspace=0.3, top=0.9, bottom=0.05, left=0.05, right=0.95)
plt.show()